# DCAR — Demographic & Contextual Anxiety Risk Model

**Component 4 · R26-DS-012** — *A Multimodal Digital Biomarker Framework for Personalized Vulnerability Mapping and Acute Escalation Forecasting in Young Adults with Anxiety Disorders*

Repo: `https://github.com/dulhara79/R26-DS-012`

---

## What this notebook produces

A single artefact bundle that, given five demographic fields collected once at patient enrolment, returns a **calibrated probability of clinically significant anxiety** — operationalised as GAD-7 total ≥ 10 — plus the uncertainty metadata the fusion layer needs.

```
{ "score": 0.41, "confidence": 0.28, "coverage": 1.0,
  "severity_probs": {...}, "expected_gad7": 8.7, "model_version": "dcar-v1.0" }
```

## The four decisions that shape everything below

**1. The target is the binary GAD-7 ≥ 10 cut-off, estimated through an ordinal model.**
GAD-7 ≥ 10 is the validated screening threshold for clinically significant generalised anxiety — it is the number a psychiatrist already reasons with, so a probability attached to it is interpretable without training. But collapsing straight to binary throws away the ordering in the four severity bands. So we fit a **cumulative ordinal model** (three linked binary models at ≥5, ≥10, ≥15), which gives us the full severity distribution *and* `P(≥10)` as one of its own outputs. We get interpretability and the clinical anchor without choosing between them.

**2. The five features are exactly what the patient app collects.**
`gender, age, edu, smoke, drink`. A model needing a variable your own app does not capture is undeployable, so deployment parity is the selection rule, not statistical appeal.

**3. `time1..time7` are excluded.**
They are response latencies recorded *during* GAD-7 administration. Using them would change the research question from "can demographics estimate anxiety burden" to "can response behaviour during the instrument predict its own score" — a different construct, measured concurrently with the target, and not collected by your app. Section 12 runs it as a labelled exploratory aside only.

**4. Expect modest performance, and frame it correctly.**
Sociodemographic variables explain a small fraction of GAD-7 variance. Realistic range here is **AUROC 0.60–0.70**. That is a legitimate result *if* this model is framed as a **population prior** — the expected anxiety burden for someone with this profile, before we look at their physiology or their notes — and a poor one if framed as a diagnostic classifier. Every number below is reported against a permutation null so the reader can see the model is doing something rather than nothing.

## Population caveat you must state in the paper

The Zenodo cohort is a general/student online sample. Your deployment population is **NHSL psychiatric inpatients and OPD attendees with a diagnosed anxiety disorder** — a fundamentally different base rate and case mix. The model transfers as a *shape* (which profiles carry more risk), not as an *absolute probability*. Section 14 exports the calibration artefacts specifically so the fusion service can re-calibrate on site data once you have 40+ NHSL patients. Do not skip that step and do not present raw Zenodo probabilities as NHSL probabilities.

## 1 · Configuration

Everything site-specific lives here so no path or constant is buried in the code below. `SYNTHETIC_FALLBACK` lets the notebook run end-to-end before you have downloaded the data, so you can verify the pipeline works and then swap in the real CSVs — it prints a loud warning and every downstream result is meaningless until you turn it off.

In [ ]:
from pathlib import Path

# ── paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = Path("data")
DEMO_CSV   = DATA_DIR / "demographic.csv"
GAD7_CSV   = DATA_DIR / "gad7.csv"
ARTEFACTS  = Path("artefacts"); ARTEFACTS.mkdir(exist_ok=True, parents=True)

# ── schema ───────────────────────────────────────────────────────────────────
ID_COL     = "export_id"
FEATURES   = ["gender", "age", "edu", "smoke", "drink"]
ITEM_COLS  = [f"question{i}" for i in range(1, 8)]
TIME_COLS  = [f"time{i}"     for i in range(1, 8)]

# ── clinical constants ───────────────────────────────────────────────────────
# GAD-7 severity bands (Spitzer et al., 2006). CUTOFF = 10 is the validated
# screening threshold for clinically significant generalised anxiety.
CUTOFF     = 10
BAND_EDGES = [4, 9, 14]              # Minimal 0-4 | Mild 5-9 | Moderate 10-14 | Severe 15-21
BAND_NAMES = ["Minimal", "Mild", "Moderate", "Severe"]
ORDINAL_THRESHOLDS = [5, 10, 15]     # cumulative model cut-points P(total >= t)

# ── protocol ─────────────────────────────────────────────────────────────────
RANDOM_STATE   = 42
TEST_FRAC      = 0.20                # touched exactly once, at the end
CALIB_FRAC     = 0.20                # held out from train, used only for calibration
N_PERMUTATIONS = 200                 # permutation null
N_BOOTSTRAP    = 1000                # CIs on test metrics
MODEL_VERSION  = "dcar-v1.0"

SYNTHETIC_FALLBACK = True            # <- set False once the real CSVs are in ./data

In [ ]:
import warnings, json, hashlib, sys
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 60)

print("python  ", sys.version.split()[0])
print("pandas  ", pd.__version__)
import sklearn; print("sklearn ", sklearn.__version__)

## 2 · Load and validate the schema

We do **not** trust column names. The loader asserts the schema and prints what it found, because a silently renamed column is the single most common cause of a research pipeline producing confident nonsense.

In [ ]:
def _synthesise(n=6000, seed=RANDOM_STATE):
    """Structure-matched stand-in so the pipeline is runnable before data arrives.
    Effect sizes are deliberately small and realistic — nothing here is a finding."""
    rng = np.random.default_rng(seed)
    gender = rng.choice(["female", "male", "other"], n, p=[0.62, 0.36, 0.02])
    age    = np.clip(rng.normal(24, 6, n), 17, 65).round(1)
    edu    = rng.choice(
        ["high school", "some college", "bachelor's degree", "master's degree", "doctorate"],
        n, p=[0.18, 0.22, 0.42, 0.15, 0.03])
    smoke  = rng.choice(
        ["never smokes", "used to smoke", "smokes occasionally", "smokes regularly"],
        n, p=[0.72, 0.10, 0.12, 0.06])
    drink  = rng.choice(
        ["never drinks", "drinks occasionally (less than once a week)",
         "drinks weekly", "drinks daily"], n, p=[0.48, 0.36, 0.13, 0.03])

    # latent anxiety trait -> graded item responses (mirrors how a real scale behaves,
    # so items are positively correlated and the total is right-skewed)
    theta = (0.40 * (gender == "female")
             - 0.025 * (age - 24)
             + 0.30 * np.isin(smoke, ["smokes occasionally", "smokes regularly"])
             + 0.22 * np.isin(drink, ["drinks weekly", "drinks daily"])
             - 0.15 * np.isin(edu, ["master's degree", "doctorate"])
             + rng.normal(0, 1.0, n)) - 0.9

    cuts = np.array([0.4, 1.5, 2.6])                    # thresholds for item response 1, 2, 3
    items = np.zeros((n, 7), dtype=int)
    for j in range(7):
        z = theta + rng.normal(0, 0.75, n)              # item-specific noise
        items[:, j] = (z[:, None] > cuts[None, :]).sum(1)
    total = items.sum(1)

    ids = np.arange(60000, 60000 + n)
    demo = pd.DataFrame({ID_COL: ids, "gender": gender, "age": age,
                         "edu": edu, "smoke": smoke, "drink": drink})
    gad = pd.DataFrame({ID_COL: ids, "score": items.sum(1)})
    for j in range(7):
        gad[f"question{j+1}"] = items[:, j]
        gad[f"time{j+1}"]     = np.round(rng.lognormal(0.6, 0.7, n), 2)
    return demo, gad


if DEMO_CSV.exists() and GAD7_CSV.exists():
    demo_raw = pd.read_csv(DEMO_CSV)
    gad_raw  = pd.read_csv(GAD7_CSV)
    USING_SYNTHETIC = False
elif SYNTHETIC_FALLBACK:
    print("!" * 78)
    print("!!  SYNTHETIC DATA IN USE — every number below is a pipeline check, not a result.")
    print("!!  Place demographic.csv and gad7.csv in ./data and set SYNTHETIC_FALLBACK=False.")
    print("!" * 78, "\n")
    demo_raw, gad_raw = _synthesise()
    USING_SYNTHETIC = True
else:
    raise FileNotFoundError(f"Expected {DEMO_CSV} and {GAD7_CSV}")

print("demographic:", demo_raw.shape, "->", list(demo_raw.columns))
print("gad7       :", gad_raw.shape,  "->", list(gad_raw.columns)[:12], "...")

In [ ]:
# ── schema assertions ────────────────────────────────────────────────────────
missing_demo = [c for c in [ID_COL] + FEATURES if c not in demo_raw.columns]
missing_gad  = [c for c in [ID_COL] + ITEM_COLS if c not in gad_raw.columns]
assert not missing_demo, f"demographic.csv missing: {missing_demo}"
assert not missing_gad,  f"gad7.csv missing: {missing_gad}"

print("Category levels found (fix the ordinal maps in §4 if any of these surprise you):\n")
for c in ["gender", "edu", "smoke", "drink"]:
    vc = demo_raw[c].astype("string").str.strip().str.lower().value_counts(dropna=False)
    print(f"── {c}  ({vc.shape[0]} levels)")
    print(vc.to_string(), "\n")

print("age  :", demo_raw["age"].describe().round(2).to_dict())

## 3 · Integrity checks and target construction

Three things get verified before a single model is fitted, because each is a silent-failure mode:

1. **Item ranges.** Every GAD-7 item must be 0–3. Anything else means a coding error or a different instrument.
2. **`score` == Σ items.** The CSV carries a `score` column *and* the seven items. If they disagree we must know which to trust. We recompute the total from items and use the recomputed value — item-level data is the primary record, a stored total is a derived convenience.
3. **Duplicate `export_id`.** A repeated id would leak the same person across train and test, inflating everything.

In [ ]:
gad = gad_raw.copy()

# 1 · item ranges
item_ok = gad[ITEM_COLS].apply(lambda s: s.between(0, 3) | s.isna()).all(axis=1)
print(f"rows with all items in [0,3]           : {item_ok.sum():,} / {len(gad):,}")
gad = gad.loc[item_ok].copy()

# 2 · recompute the total; compare against the stored `score`
gad["gad7_total"] = gad[ITEM_COLS].sum(axis=1)
if "score" in gad.columns:
    mismatch = (gad["score"] != gad["gad7_total"]).sum()
    print(f"stored `score` != recomputed sum        : {mismatch:,} "
          f"({mismatch/max(len(gad),1):.2%})  -> using the recomputed sum")
assert gad["gad7_total"].between(0, 21).all()

# 3 · duplicates
dupes = gad[ID_COL].duplicated().sum() + demo_raw[ID_COL].duplicated().sum()
print(f"duplicate ids (gad + demo)              : {dupes}")
gad  = gad.drop_duplicates(subset=[ID_COL], keep="first")
demo = demo_raw.drop_duplicates(subset=[ID_COL], keep="first")

# ── merge ────────────────────────────────────────────────────────────────────
df = demo[[ID_COL] + FEATURES].merge(
    gad[[ID_COL, "gad7_total"] + ITEM_COLS + [c for c in TIME_COLS if c in gad.columns]],
    on=ID_COL, how="inner")
print(f"\nmerged cohort                           : {len(df):,} participants")
print(f"lost from demographic (no GAD-7)        : {len(demo) - len(df):,}")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# targets
df["severity"] = pd.cut(df["gad7_total"], bins=[-1] + BAND_EDGES + [21],
                        labels=BAND_NAMES).astype(str)
df["y"] = (df["gad7_total"] >= CUTOFF).astype(int)

print("GAD-7 total  ", df["gad7_total"].describe().round(2).to_dict())
print("\nseverity distribution:")
print(df["severity"].value_counts().reindex(BAND_NAMES).to_frame("n")
      .assign(pct=lambda d: (d.n / len(df) * 100).round(1)).to_string())
print(f"\nprevalence of GAD-7 >= {CUTOFF}: {df['y'].mean():.3f}  "
      f"(n_pos = {df['y'].sum():,})")

# internal consistency of the instrument in THIS cohort — reported in the paper
items = df[ITEM_COLS].astype(float)
k, iv, tv = items.shape[1], items.var(ddof=1), items.sum(axis=1).var(ddof=1)
print(f"Cronbach's alpha (GAD-7, this cohort): {(k/(k-1))*(1 - iv.sum()/tv):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
df["gad7_total"].plot.hist(bins=22, ax=ax[0], edgecolor="white")
ax[0].axvline(CUTOFF, color="crimson", ls="--"); ax[0].set_title("GAD-7 total"); ax[0].set_xlabel("score")
df["severity"].value_counts().reindex(BAND_NAMES).plot.bar(ax=ax[1], rot=0)
ax[1].set_title("severity band")
plt.tight_layout(); plt.savefig(ARTEFACTS / "fig_target_distribution.png", dpi=140); plt.show()

### A note on what the histogram is telling you

If the distribution is heavily zero-inflated — a large spike at 0 with a thin tail — that is characteristic of general-population GAD-7 samples and it has two consequences you must handle rather than ignore:

- **Accuracy becomes meaningless.** Predicting "not anxious" for everyone will look excellent. That is why every metric below is AUROC / AUPRC / balanced accuracy, and why the prevalence baseline is reported alongside.
- **The positive class is small**, so confidence intervals will be wide. We bootstrap all test metrics rather than reporting bare point estimates.

## 4 · Encoding — and why each choice

| Variable | Treatment | Reason |
|---|---|---|
| `gender` | one-hot, nominal | No defensible ordering. Forcing one would impose a false monotone effect. |
| `age` | natural cubic spline, 4 knots | Anxiety–age is well documented as **non-monotone** (peaks in young adulthood, declines later). A linear term would average that curve away and understate the effect in exactly your target group. |
| `edu` | ordinal integer | Genuinely ordered; ordinal coding spends 1 parameter instead of *k*−1, which matters for stable coefficients and for a cohort where higher levels are sparse. |
| `smoke` | ordinal integer | Ordered by consumption intensity. |
| `drink` | ordinal integer | Ordered by frequency. |
| unmapped / missing | explicit `Unknown` level + indicator column | Never silently impute a category. Missingness in a self-report profile is itself informative, and the fusion layer needs the coverage number. |

The maps below use substring matching so they survive the exact wording in the CSV. **Any level printed as UNMAPPED must be fixed before you continue** — an unmapped level silently becomes the neutral midpoint, which is a quiet way to destroy signal.

In [ ]:
EDU_ORDER = [
    (["no formal", "primary", "less than high"],                       0),
    (["high school", "secondary", "o/l", "ordinary level"],             1),
    (["some college", "diploma", "a/l", "advanced level", "associate"], 2),
    (["bachelor", "undergrad", "b.sc", "bsc"],                          3),
    (["master", "m.sc", "msc", "postgrad"],                             4),
    (["doctor", "phd", "ph.d"],                                         5),
]
SMOKE_ORDER = [
    (["never"],                                  0),
    (["used to", "former", "ex-"],               1),
    (["occasion", "sometimes", "rarely"],        2),
    (["regular", "daily", "every day", "heavy"], 3),
]
DRINK_ORDER = [
    (["never"],                                              0),
    (["occasion", "less than once a week", "rarely", "monthly"], 1),
    (["week", "several times"],                              2),
    (["daily", "every day"],                                 3),
]

def ordinal_encode(series, ruleset, name):
    s = series.astype("string").str.strip().str.lower()
    out = pd.Series(np.nan, index=s.index, dtype="float")
    for keys, val in ruleset:
        hit = s.apply(lambda v: isinstance(v, str) and any(k in v for k in keys))
        out.loc[hit & out.isna()] = val
    unmapped = sorted(set(s[out.isna() & s.notna()].dropna()))
    if unmapped:
        print(f"  !! UNMAPPED in `{name}` -> {unmapped}   (add these to the rules above)")
    return out

enc = pd.DataFrame(index=df.index)
print("Ordinal encoding:")
enc["edu_ord"]   = ordinal_encode(df["edu"],   EDU_ORDER,   "edu")
enc["smoke_ord"] = ordinal_encode(df["smoke"], SMOKE_ORDER, "smoke")
enc["drink_ord"] = ordinal_encode(df["drink"], DRINK_ORDER, "drink")

for c in ["edu_ord", "smoke_ord", "drink_ord"]:
    enc[c + "_missing"] = enc[c].isna().astype(int)
    enc[c] = enc[c].fillna(enc[c].median())

enc["age"]          = pd.to_numeric(df["age"], errors="coerce")
enc["age_missing"]  = enc["age"].isna().astype(int)
enc["age"]          = enc["age"].fillna(enc["age"].median())

g = df["gender"].astype("string").str.strip().str.lower().fillna("unknown")
g = g.where(g.isin(["female", "male"]), "other_unknown")
enc = pd.concat([enc, pd.get_dummies(g, prefix="gender").astype(int)], axis=1)

for col in ["gender_female", "gender_male", "gender_other_unknown"]:
    if col not in enc.columns:
        enc[col] = 0

X = enc.copy()
y = df["y"].values
y_total = df["gad7_total"].values
print(f"\ndesign matrix: {X.shape}  ->  {list(X.columns)}")

## 5 · Splits

Three disjoint partitions, stratified on the binary target:

- **train (60%)** — model fitting and cross-validated selection
- **calibration (20%)** — *only* for fitting the probability calibrator. Calibrating on training predictions produces optimistically sharp probabilities, and probability quality is the whole point of this model, since a miscalibrated `r_demo` corrupts the fusion layer downstream.
- **test (20%)** — touched exactly once, at the end

One row per participant, so a simple stratified split is sufficient. If you later add the COVID-19 Citizen Science cohort with its repeated measures, this must become `StratifiedGroupKFold` on participant id.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict

X_dev, X_test, y_dev, y_test, tot_dev, tot_test, idx_dev, idx_test = train_test_split(
    X, y, y_total, df.index, test_size=TEST_FRAC, stratify=y, random_state=RANDOM_STATE)

rel_calib = CALIB_FRAC / (1 - TEST_FRAC)
X_tr, X_cal, y_tr, y_cal, tot_tr, tot_cal = train_test_split(
    X_dev, y_dev, tot_dev, test_size=rel_calib, stratify=y_dev, random_state=RANDOM_STATE)

for nm, yy in [("train", y_tr), ("calib", y_cal), ("test", y_test)]:
    print(f"{nm:6s} n={len(yy):6,d}   prevalence={yy.mean():.3f}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 6 · Metrics

`ECE` and `expected_calibration_error` are not in sklearn, so they are defined here. Calibration is reported as prominently as discrimination — an AUROC of 0.68 with ECE 0.02 is a usable fusion input; AUROC 0.70 with ECE 0.15 is not.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             balanced_accuracy_score, f1_score, confusion_matrix,
                             cohen_kappa_score, roc_curve)

def ece(y_true, p, n_bins=10):
    """Expected calibration error, equal-width bins."""
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum():
            e += m.mean() * abs(y_true[m].mean() - p[m].mean())
    return e

def binary_report(y_true, p, label="", thr=None):
    thr = thr if thr is not None else 0.5
    yhat = (p >= thr).astype(int)
    return {
        "model": label,
        "AUROC": roc_auc_score(y_true, p),
        "AUPRC": average_precision_score(y_true, p),
        "Brier": brier_score_loss(y_true, p),
        "ECE":   ece(np.asarray(y_true), np.asarray(p)),
        "BalAcc": balanced_accuracy_score(y_true, yhat),
        "F1":    f1_score(y_true, yhat, zero_division=0),
    }

def bootstrap_ci(y_true, p, fn=roc_auc_score, n=N_BOOTSTRAP, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_true, p = np.asarray(y_true), np.asarray(p)
    out = []
    for _ in range(n):
        i = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[i])) < 2:
            continue
        out.append(fn(y_true[i], p[i]))
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

## 7 · Baselines and the permutation null

Two reference points, both mandatory for a paper:

- **Prevalence baseline** — predict the base rate for everyone. AUROC 0.5 by construction; it anchors Brier and AUPRC.
- **Permutation null** — refit the real model on labels shuffled within the training set, 200 times. The resulting AUROC distribution is what this pipeline produces *from noise*. Your headline number must clear it, and the gap is the finding.

This matters more here than in a typical study: with weak demographic signal and a 5-variable design matrix, an AUROC of 0.58 is very close to what an unregularised pipeline can manufacture by chance. Reporting the null is what separates a defensible weak result from an undetected artefact.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SPLINE_COLS = ["age"]
LINEAR_COLS = [c for c in X.columns if c != "age"]

def make_glm(C=1.0):
    """Logistic regression with a natural cubic spline on age.
    Spline on age because the age-anxiety relationship is non-monotone; linear
    everywhere else because the remaining features are ordinal or binary."""
    pre = ColumnTransformer([
        ("age_spline", Pipeline([("sc", StandardScaler()),
                                 ("sp", SplineTransformer(n_knots=4, degree=3,
                                                          include_bias=False))]), SPLINE_COLS),
        ("lin", StandardScaler(), LINEAR_COLS),
    ])
    return Pipeline([("pre", pre),
                     ("clf", LogisticRegression(C=C, max_iter=2000,
                                                class_weight="balanced",
                                                random_state=RANDOM_STATE))])

dummy = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
p_dummy = dummy.predict_proba(X_test)[:, 1]
print("prevalence baseline  Brier =", round(brier_score_loss(y_test, p_dummy), 4))

In [ ]:
# in-development CV estimate for the primary model (no test-set contact)
glm = make_glm()
p_cv = cross_val_predict(glm, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
auc_cv = roc_auc_score(y_tr, p_cv)
print(f"GLM  cross-validated AUROC (train) = {auc_cv:.4f}")

rng = np.random.default_rng(RANDOM_STATE)
null = []
for i in range(N_PERMUTATIONS):
    y_perm = rng.permutation(y_tr)
    p_perm = cross_val_predict(make_glm(), X_tr, y_perm, cv=3, method="predict_proba")[:, 1]
    null.append(roc_auc_score(y_perm, p_perm))
null = np.array(null)
p_value = (np.sum(null >= auc_cv) + 1) / (len(null) + 1)

print(f"permutation null AUROC: mean={null.mean():.4f}  sd={null.std():.4f}  "
      f"95th pct={np.percentile(null,95):.4f}")
print(f"observed {auc_cv:.4f}  ->  p = {p_value:.4f}"
      f"   {'CLEARS the null' if p_value < 0.05 else 'DOES NOT clear the null'}")

plt.figure(figsize=(6, 2.8))
plt.hist(null, bins=30, color="#9db4c0", edgecolor="white")
plt.axvline(auc_cv, color="crimson", lw=2, label=f"observed {auc_cv:.3f}")
plt.xlabel("AUROC under permuted labels"); plt.legend(); plt.tight_layout()
plt.savefig(ARTEFACTS / "fig_permutation_null.png", dpi=140); plt.show()

## 8 · Model comparison

Four candidates, in increasing complexity. The point is not to find a winner; it is to show whether complexity earned anything on a five-variable problem. With this feature space — three low-cardinality ordinals, one binary-ish nominal, one continuous — the effective number of distinct patient profiles is small, so a well-regularised GLM is genuinely competitive with gradient boosting. If the GLM ties, **pick the GLM**: it is interpretable, its coefficients convert directly to odds ratios a clinician can read, and it extrapolates more sanely to a new population than a tree ensemble that has memorised the Zenodo profile grid.

In [ ]:
candidates = {
    "GLM (spline age, C=1.0)":  make_glm(1.0),
    "GLM (strong reg, C=0.1)":  make_glm(0.1),
    "HistGradientBoosting":     HistGradientBoostingClassifier(
                                    max_depth=3, max_iter=250, learning_rate=0.06,
                                    l2_regularization=1.0, random_state=RANDOM_STATE),
}

rows = []
for name, mdl in candidates.items():
    p = cross_val_predict(mdl, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    r = binary_report(y_tr, p, name); r["split"] = "train-CV"
    rows.append(r)

comparison = pd.DataFrame(rows).set_index("model").round(4)
print(comparison.to_string())

BEST_NAME = comparison["AUROC"].idxmax()
print(f"\nbest by CV AUROC: {BEST_NAME}")
print("If the GLM is within ~0.005 AUROC of the boosted model, override this and take the GLM.")
BEST = candidates[BEST_NAME]

## 9 · The ordinal model — full severity distribution

The binary model gives `P(≥10)`. The ordinal model gives the whole picture: `P(≥5)`, `P(≥10)`, `P(≥15)`, from which the four band probabilities follow by differencing.

We use the **cumulative-link decomposition**: three independent binary classifiers, one per threshold. Monotonicity (`P(≥5) ≥ P(≥10) ≥ P(≥15)`) is not guaranteed by independent fits, so it is enforced afterwards by a running minimum. This is deliberately chosen over a single proportional-odds model because the **proportional-odds assumption is frequently violated** for GAD-7 — the effect of, say, gender on crossing the *mild* threshold is not the same as on crossing the *severe* threshold — and independent fits let each threshold have its own coefficients.

The `≥10` head of this model is what feeds the fusion layer, so the binary and ordinal outputs are guaranteed consistent — they are the same fitted object.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone

class CumulativeOrdinal(BaseEstimator, ClassifierMixin):
    """P(total >= t) for each t in thresholds, monotonicity enforced post-hoc."""
    def __init__(self, base=None, thresholds=(5, 10, 15)):
        self.base, self.thresholds = base, thresholds

    MIN_POS = 25          # below this a threshold cannot be fitted responsibly

    def fit(self, X, totals):
        self.models_, self.constant_ = {}, {}
        for t in self.thresholds:
            yt = (np.asarray(totals) >= t).astype(int)
            if yt.sum() < self.MIN_POS or yt.sum() == len(yt):
                # Degenerate threshold (e.g. almost nobody scores >=15 in this cohort).
                # Fall back to the empirical rate rather than fitting noise, and say so.
                self.constant_[t] = float(yt.mean())
                print(f"  !! threshold >={t}: only {int(yt.sum())} positives "
                      f"-> using empirical rate {yt.mean():.4f}, not a fitted model")
                continue
            m = clone(self.base) if self.base is not None else make_glm()
            self.models_[t] = m.fit(X, yt)
        assert CUTOFF in self.models_, (
            f"The primary threshold >={CUTOFF} could not be fitted — too few positives. "
            "Check the cohort or lower CUTOFF to 5.")
        return self

    def _col(self, X, t):
        if t in self.models_:
            return self.models_[t].predict_proba(X)[:, 1]
        return np.full(len(X), self.constant_[t])

    def cumulative(self, X):
        P = np.column_stack([self._col(X, t) for t in self.thresholds])
        return np.minimum.accumulate(P, axis=1)          # enforce P(>=5) >= P(>=10) >= P(>=15)

    def band_proba(self, X):
        """-> columns [Minimal, Mild, Moderate, Severe]"""
        P = self.cumulative(X)
        return np.column_stack([1 - P[:, 0], P[:, 0] - P[:, 1], P[:, 1] - P[:, 2], P[:, 2]]).clip(0, 1)

    def predict_proba(self, X):                          # binary head at CUTOFF
        p = self.cumulative(X)[:, list(self.thresholds).index(CUTOFF)]
        return np.column_stack([1 - p, p])

    def expected_total(self, X):
        """Band midpoints weighted by band probability — a readable GAD-7 estimate."""
        return self.band_proba(X) @ np.array([2.0, 7.0, 12.0, 18.0])


ordinal = CumulativeOrdinal(base=make_glm(1.0), thresholds=tuple(ORDINAL_THRESHOLDS))
ordinal.fit(X_tr, tot_tr)

p_ord_cal = ordinal.predict_proba(X_cal)[:, 1]
print("ordinal >=10 head, calibration split:",
      {k: round(v, 4) for k, v in binary_report(y_cal, p_ord_cal, "ordinal").items() if k != "model"})

bp = ordinal.band_proba(X_cal)
print("\nmean predicted band probabilities:", dict(zip(BAND_NAMES, bp.mean(0).round(3))))
print("observed band frequencies         :",
      df.loc[X_cal.index, "severity"].value_counts(normalize=True).reindex(BAND_NAMES).round(3).to_dict())

## 10 · Calibration

The fusion layer combines `r_demo` with probabilities from three other models. If `r_demo` is systematically over-confident, the fusion weight assigned to it is meaningless — a weight multiplies a probability, so a wrong probability corrupts the composite no matter how well the weight is tuned. **Calibration is therefore a fusion requirement, not a nicety.**

Isotonic regression, fitted on the held-out calibration split. Isotonic over Platt because it is non-parametric and the miscalibration in a class-weighted logistic model is typically not a simple sigmoid distortion; with a few thousand calibration rows there is enough data for isotonic not to overfit.

> **Calibrate every threshold, not just the binary head.** This is easy to get wrong and the failure is silent. `class_weight="balanced"` deliberately inflates predicted probabilities for the rare class — that is what makes the model discriminate — so the *raw* cumulative outputs sit near 0.5 for everyone. Calibrating only `≥10` fixes the fusion score but leaves `P(≥5)` and `P(≥15)` on the inflated scale, and the band probabilities computed by differencing them come out as nonsense: in the first run of this notebook a low-risk 21-year-old was assigned `P(Severe) = 0.47`. The clinician-facing severity distribution would have been garbage while the headline AUROC looked fine. Each threshold gets its own isotonic map, and monotonicity is re-enforced afterwards.

In [ ]:
from sklearn.isotonic import IsotonicRegression

class CalibratedOrdinal:
    """Per-threshold isotonic calibration on top of the cumulative ordinal model."""
    def __init__(self, ordinal, thresholds):
        self.ordinal, self.thresholds = ordinal, list(thresholds)
        self.isos_ = {}

    def fit(self, X_cal, totals_cal):
        for i, t in enumerate(self.thresholds):
            raw = self.ordinal._col(X_cal, t)
            yt = (np.asarray(totals_cal) >= t).astype(int)
            if yt.sum() < 10 or len(np.unique(raw)) < 3:
                self.isos_[t] = None                       # nothing to learn; pass through
                continue
            self.isos_[t] = IsotonicRegression(out_of_bounds="clip",
                                               y_min=0.0, y_max=1.0).fit(raw, yt)
        return self

    def cumulative(self, X):
        cols = []
        for t in self.thresholds:
            raw = self.ordinal._col(X, t)
            cols.append(raw if self.isos_[t] is None else self.isos_[t].predict(raw))
        return np.minimum.accumulate(np.column_stack(cols), axis=1)

    def band_proba(self, X):
        P = self.cumulative(X)
        return np.column_stack([1 - P[:, 0], P[:, 0] - P[:, 1],
                                P[:, 1] - P[:, 2], P[:, 2]]).clip(0, 1)

    def proba(self, X):
        return self.cumulative(X)[:, self.thresholds.index(CUTOFF)]

    def expected_total(self, X):
        return self.band_proba(X) @ np.array([2.0, 7.0, 12.0, 18.0])


DCAR = CalibratedOrdinal(ordinal, ORDINAL_THRESHOLDS).fit(X_cal, tot_cal)

def dcar_proba(Xq):
    """Final calibrated P(GAD-7 >= 10) — the score the fusion layer consumes."""
    return DCAR.proba(Xq)

pre_cal  = ordinal.predict_proba(X_test)[:, 1]
post_cal = dcar_proba(X_test)

# sanity: calibrated band probabilities must track observed band frequencies
print("mean predicted bands (calibrated):",
      dict(zip(BAND_NAMES, DCAR.band_proba(X_test).mean(0).round(3))))
print("observed band frequencies       :",
      df.loc[X_test.index, "severity"].value_counts(normalize=True)
        .reindex(BAND_NAMES).fillna(0).round(3).to_dict())

print(pd.DataFrame([
    binary_report(y_test, pre_cal,  "ordinal, uncalibrated"),
    binary_report(y_test, post_cal, "ordinal, isotonic-calibrated"),
]).set_index("model").round(4).to_string())

# reliability diagram
def reliability(y_true, p, ax, label):
    edges = np.linspace(0, 1, 11); idx = np.clip(np.digitize(p, edges[1:-1]), 0, 9)
    xs, ys = [], []
    for b in range(10):
        m = idx == b
        if m.sum() > 10:
            xs.append(p[m].mean()); ys.append(np.asarray(y_true)[m].mean())
    ax.plot(xs, ys, "o-", label=label)

fig, ax = plt.subplots(figsize=(4.4, 4.2))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
reliability(y_test, pre_cal, ax, "uncalibrated")
reliability(y_test, post_cal, ax, "calibrated")
ax.set_xlabel("predicted P(GAD-7 >= 10)"); ax.set_ylabel("observed frequency")
ax.legend(); plt.tight_layout(); plt.savefig(ARTEFACTS / "fig_reliability.png", dpi=140); plt.show()

## 11 · Test-set evaluation

The test split is used here for the first and only time. Everything is reported with bootstrap confidence intervals, because with a small positive class a bare point estimate overstates what you know.

**Operating threshold.** Not 0.5. In a psychiatric setting the cost of missing a moderately anxious patient exceeds the cost of a false flag that a clinician dismisses in two seconds. We select the threshold on the **calibration split** (never the test split) to maximise F2 — recall weighted twice as heavily as precision — and then freeze it.

In [ ]:
from sklearn.metrics import fbeta_score

# threshold chosen on the calibration split only
p_cal_final = dcar_proba(X_cal)
grid = np.linspace(0.02, 0.95, 187)
f2 = [fbeta_score(y_cal, (p_cal_final >= t).astype(int), beta=2, zero_division=0) for t in grid]
THRESHOLD = float(grid[int(np.argmax(f2))])
print(f"operating threshold (max F2 on calibration split) = {THRESHOLD:.3f}")
if THRESHOLD <= grid[1] or THRESHOLD >= grid[-2]:
    print("  !! threshold sits at the edge of the search grid — F2 is being maximised by\n     flagging (almost) everyone. Inspect the precision-recall curve before adopting it.")

res = binary_report(y_test, post_cal, "DCAR v1.0", thr=THRESHOLD)
lo, hi = bootstrap_ci(y_test, post_cal, roc_auc_score)
lo_ap, hi_ap = bootstrap_ci(y_test, post_cal, average_precision_score)

print("\n── TEST SET ─────────────────────────────────────────────")
print(f"AUROC   {res['AUROC']:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
print(f"AUPRC   {res['AUPRC']:.4f}   95% CI [{lo_ap:.4f}, {hi_ap:.4f}]   (prevalence {y_test.mean():.3f})")
print(f"Brier   {res['Brier']:.4f}")
print(f"ECE     {res['ECE']:.4f}")
print(f"BalAcc  {res['BalAcc']:.4f}  @ thr={THRESHOLD:.3f}")
print(f"F1      {res['F1']:.4f}")

tn, fp, fn_, tp = confusion_matrix(y_test, (post_cal >= THRESHOLD).astype(int)).ravel()
print(f"\nsensitivity {tp/max(tp+fn_,1):.3f}   specificity {tn/max(tn+fp,1):.3f}   "
      f"PPV {tp/max(tp+fp,1):.3f}   NPV {tn/max(tn+fn_,1):.3f}")

# ordinal quality on the 4-band task
bands_pred = np.array(BAND_NAMES)[DCAR.band_proba(X_test).argmax(1)]
bands_true = df.loc[X_test.index, "severity"].values
order = {b: i for i, b in enumerate(BAND_NAMES)}
qwk = cohen_kappa_score([order[b] for b in bands_true], [order[b] for b in bands_pred],
                        weights="quadratic")
print(f"\n4-band quadratic-weighted kappa: {qwk:.4f}   macro-F1: "
      f"{f1_score(bands_true, bands_pred, average='macro', zero_division=0):.4f}")
print(f"expected-GAD7 MAE: {np.abs(DCAR.expected_total(X_test) - tot_test).mean():.2f} points")

## 12 · Ablations and subgroup performance

**Feature ablation** answers your "5 variables" question empirically. Each feature is removed, the model refitted, and the AUROC drop reported. If `smoke` costs nothing, drop it — a documented ablation is a far better justification for a four-variable model than a preference.

**Subgroup performance** is mandatory, not optional, for a model whose inputs are gender, age and education. A model that systematically under-predicts risk in one group is a safety problem, and finding it is a contribution.

In [ ]:
FEATURE_GROUPS = {
    "gender": [c for c in X.columns if c.startswith("gender_")],
    "age":    ["age", "age_missing"],
    "edu":    ["edu_ord", "edu_ord_missing"],
    "smoke":  ["smoke_ord", "smoke_ord_missing"],
    "drink":  ["drink_ord", "drink_ord_missing"],
}

def refit_auc(cols):
    Xtr2, Xte2 = X_tr[cols], X_test[cols]
    global SPLINE_COLS, LINEAR_COLS
    sp_bak, ln_bak = SPLINE_COLS, LINEAR_COLS
    SPLINE_COLS = [c for c in ["age"] if c in cols]
    LINEAR_COLS = [c for c in cols if c != "age"]
    try:
        m = CumulativeOrdinal(base=make_glm(1.0), thresholds=tuple(ORDINAL_THRESHOLDS)).fit(Xtr2, tot_tr)
        c = CalibratedOrdinal(m, ORDINAL_THRESHOLDS).fit(X_cal[cols], tot_cal)
        return roc_auc_score(y_test, c.proba(Xte2))
    finally:
        SPLINE_COLS, LINEAR_COLS = sp_bak, ln_bak

full_auc = refit_auc(list(X.columns))
abl = [{"removed": "— none (full model)", "AUROC": full_auc, "delta": 0.0}]
for name, cols in FEATURE_GROUPS.items():
    keep = [c for c in X.columns if c not in cols]
    a = refit_auc(keep)
    abl.append({"removed": name, "AUROC": a, "delta": a - full_auc})
abl_df = pd.DataFrame(abl).set_index("removed").round(4)
print(abl_df.to_string())
print("\nA `delta` near 0 means that feature contributes nothing measurable -> justify dropping it.")

In [ ]:
sub = pd.DataFrame({"p": post_cal, "y": y_test,
                    "gender": df.loc[X_test.index, "gender"].astype(str).str.lower().values,
                    "age_band": pd.cut(df.loc[X_test.index, "age"],
                                       [0, 21, 25, 30, 200],
                                       labels=["<=21", "22-25", "26-30", "30+"]).astype(str)})
rows = []
for key in ["gender", "age_band"]:
    for lvl, gdf in sub.groupby(key):
        if gdf["y"].nunique() < 2 or len(gdf) < 40:
            continue
        l, h = bootstrap_ci(gdf["y"].values, gdf["p"].values, roc_auc_score, n=400)
        rows.append({"slice": f"{key}={lvl}", "n": len(gdf), "prev": gdf["y"].mean(),
                     "AUROC": roc_auc_score(gdf["y"], gdf["p"]), "lo": l, "hi": h,
                     "ECE": ece(gdf["y"].values, gdf["p"].values)})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nReport every row of this table in the paper, including the bad ones.")

In [ ]:
# ── optional exploratory aside: response latency (NOT part of the deployed model) ──
if all(c in df.columns for c in TIME_COLS):
    from sklearn.linear_model import LogisticRegression as _LR
    Xt_tr = np.log1p(df.loc[X_tr.index, TIME_COLS].astype(float).fillna(0).values)
    Xt_te = np.log1p(df.loc[X_test.index, TIME_COLS].astype(float).fillna(0).values)
    m = Pipeline([("sc", StandardScaler()),
                  ("lr", _LR(max_iter=1000, class_weight="balanced"))]).fit(Xt_tr, y_tr)
    print(f"[EXPLORATORY] response-latency-only AUROC = "
          f"{roc_auc_score(y_test, m.predict_proba(Xt_te)[:,1]):.4f}")
    print("Reported as an aside only: paradata measured during the instrument, and not "
          "collected by the patient app -> excluded from the deployed model.")

## 13 · Interpretation

Odds ratios from the `≥10` head, plus permutation importance. Both go in the paper: odds ratios because a clinician can read them, permutation importance because it is model-agnostic and reflects the fitted model rather than the parameterisation.

In [ ]:
from sklearn.inspection import permutation_importance

head = ordinal.models_[CUTOFF]
pre  = head.named_steps["pre"]
coef = head.named_steps["clf"].coef_[0]
names = list(pre.get_feature_names_out())
odds = (pd.DataFrame({"feature": names, "coef": coef, "odds_ratio": np.exp(coef)})
          .sort_values("coef", key=abs, ascending=False).round(4))
print("Odds ratios, P(GAD-7 >= 10):\n", odds.to_string(index=False))
print("\n(Spline basis terms for age are not individually interpretable — read the age curve below.)")

pi = permutation_importance(head, X_test, y_test, n_repeats=20,
                            random_state=RANDOM_STATE, scoring="roc_auc")
imp = (pd.DataFrame({"feature": X.columns, "importance": pi.importances_mean,
                     "sd": pi.importances_std})
         .sort_values("importance", ascending=False).round(4))
print("\nPermutation importance (drop in test AUROC):\n", imp.to_string(index=False))

In [ ]:
# the age curve — the reason we used a spline
ages = np.linspace(max(17, np.nanpercentile(df["age"], 1)),
                   min(60, np.nanpercentile(df["age"], 99)), 60)
proto = X_tr.median().to_frame().T
grid = pd.concat([proto] * len(ages), ignore_index=True); grid["age"] = ages
grid = grid[X.columns]
plt.figure(figsize=(6, 3.2))
plt.plot(ages, dcar_proba(grid), lw=2)
plt.xlabel("age"); plt.ylabel("P(GAD-7 >= 10)")
plt.title("Age effect at the median profile (spline)")
plt.tight_layout(); plt.savefig(ARTEFACTS / "fig_age_curve.png", dpi=140); plt.show()

## 14 · The deployed scoring function

This is the exact function the Hugging Face Space wraps. Three outputs beyond the score, each required by the fusion layer:

- **`score`** — calibrated `P(GAD-7 ≥ 10)`.
- **`confidence`** — `1 − H(band probabilities)/log 4`. A patient profile that maps to a flat distribution across the four bands is one this model cannot resolve, and the fusion gate must down-weight it. Note that demographic confidence is *intrinsically low* — that is the honest signal, not a defect.
- **`coverage`** — fraction of the five inputs actually supplied. A profile with three of five fields present is worth less than a complete one, and the gate needs to know.

We also export the **reference score distribution** on the test split. The fusion service uses it as the percentile map in its harmonisation stage, so `r_demo` is comparable with the physiological and clinical-text scores. Without this file, the fusion layer is averaging incommensurable numbers.

In [ ]:
def build_features(record: dict) -> pd.DataFrame:
    """dict from the patient app -> one design-matrix row. Mirrors §4 exactly."""
    def _ord(val, rules):
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return np.nan
        s = str(val).strip().lower()
        for keys, v in rules:
            if any(k in s for k in keys):
                return float(v)
        return np.nan

    row = {}
    e = _ord(record.get("edu"),   EDU_ORDER)
    sm = _ord(record.get("smoke"), SMOKE_ORDER)
    dr = _ord(record.get("drink"), DRINK_ORDER)
    age = record.get("age")
    age = float(age) if age not in (None, "") else np.nan

    row["edu_ord"], row["edu_ord_missing"]     = (MEDIANS["edu_ord"]   if np.isnan(e)  else e),  int(np.isnan(e))
    row["smoke_ord"], row["smoke_ord_missing"] = (MEDIANS["smoke_ord"] if np.isnan(sm) else sm), int(np.isnan(sm))
    row["drink_ord"], row["drink_ord_missing"] = (MEDIANS["drink_ord"] if np.isnan(dr) else dr), int(np.isnan(dr))
    row["age"], row["age_missing"]             = (MEDIANS["age"]       if np.isnan(age) else age), int(np.isnan(age))

    g = str(record.get("gender", "")).strip().lower()
    g = g if g in ("female", "male") else "other_unknown"
    for lvl in ("female", "male", "other_unknown"):
        row[f"gender_{lvl}"] = int(g == lvl)
    return pd.DataFrame([row])[X.columns]


MEDIANS = X_tr.median().to_dict()

def dcar_predict(record: dict) -> dict:
    Xq = build_features(record)
    p = float(dcar_proba(Xq)[0])
    bands = DCAR.band_proba(Xq)[0]
    H = -np.sum(np.where(bands > 0, bands * np.log(bands + 1e-12), 0.0))
    supplied = sum(record.get(k) not in (None, "", float("nan")) for k in FEATURES)
    return {
        "score": round(p, 4),
        "risk_label": "elevated" if p >= THRESHOLD else "not elevated",
        "threshold": round(THRESHOLD, 4),
        "severity_probs": {b: round(float(v), 4) for b, v in zip(BAND_NAMES, bands)},
        "most_likely_band": BAND_NAMES[int(bands.argmax())],
        "expected_gad7": round(float(DCAR.expected_total(Xq)[0]), 2),
        "confidence": round(float(1 - H / np.log(len(BAND_NAMES))), 4),
        "coverage": round(supplied / len(FEATURES), 4),
        "model_version": MODEL_VERSION,
    }

demo_case = {"gender": "female", "age": 21, "edu": "bachelor's degree",
             "smoke": "never smokes", "drink": "never drinks"}
print(json.dumps(dcar_predict(demo_case), indent=2))
print()
print(json.dumps(dcar_predict({"gender": "male", "age": 34, "edu": None,
                               "smoke": "smokes regularly", "drink": "drinks daily"}), indent=2))

In [ ]:
import joblib

ref_scores = np.sort(dcar_proba(X_test))

# NOTE ON THE EXPORT FORMAT
# joblib pickles custom classes *by reference* — a bundle containing CumulativeOrdinal
# would only unpickle in a process where that class is importable, and the Hugging Face
# Space would crash on startup with AttributeError. So we export only plain sklearn
# objects (Pipelines, IsotonicRegression) plus the constants, and the Space reimplements
# the ~15 lines of cumulative arithmetic. Portable, and the serving logic stays auditable.

bundle = {
    "thresholds":          list(ORDINAL_THRESHOLDS),
    "threshold_models":    {t: ordinal.models_.get(t)    for t in ORDINAL_THRESHOLDS},
    "threshold_constants": {t: ordinal.constant_.get(t)  for t in ORDINAL_THRESHOLDS},
    "isotonics":           {t: DCAR.isos_.get(t)         for t in ORDINAL_THRESHOLDS},
    "band_midpoints":      [2.0, 7.0, 12.0, 18.0],
    "medians": MEDIANS, "columns": list(X.columns), "threshold": THRESHOLD,
    "edu_order": EDU_ORDER, "smoke_order": SMOKE_ORDER, "drink_order": DRINK_ORDER,
    "band_names": BAND_NAMES, "features": FEATURES, "cutoff": CUTOFF,
    "version": MODEL_VERSION,
    "reference_scores": ref_scores,          # <- fusion harmonisation percentile map
}
joblib.dump(bundle, ARTEFACTS / "dcar_model.joblib")

# round-trip check: the Space must reproduce the notebook's number exactly
_b = joblib.load(ARTEFACTS / "dcar_model.joblib")
_x = build_features(demo_case)
_cols = []
for t in _b["thresholds"]:
    m, c, iso_t = _b["threshold_models"][t], _b["threshold_constants"][t], _b["isotonics"][t]
    raw = m.predict_proba(_x)[:, 1] if m is not None else np.full(len(_x), c)
    _cols.append(raw if iso_t is None else iso_t.predict(raw))
_P = np.minimum.accumulate(np.column_stack(_cols), axis=1)
_reload = float(_P[0, _b["thresholds"].index(CUTOFF)])
assert abs(_reload - float(dcar_proba(_x)[0])) < 1e-9, "serving path diverges from notebook!"
print(f"round-trip OK: reloaded score {_reload:.4f}")

metadata = {
    "model_version": MODEL_VERSION,
    "target": f"P(GAD-7 total >= {CUTOFF})",
    "features": FEATURES,
    "training_cohort": "Zenodo psychological assessment dataset (10423537)",
    "n_train": int(len(y_tr)), "n_calib": int(len(y_cal)), "n_test": int(len(y_test)),
    "prevalence_test": float(y_test.mean()),
    "test_AUROC": float(res["AUROC"]), "test_AUROC_CI": [lo, hi],
    "test_AUPRC": float(res["AUPRC"]), "test_Brier": float(res["Brier"]),
    "test_ECE": float(res["ECE"]), "qwk_4band": float(qwk),
    "operating_threshold": THRESHOLD,
    "permutation_null_mean": float(null.mean()), "permutation_p": float(p_value),
    "synthetic_data_used": bool(USING_SYNTHETIC),
    "population_caveat": ("Trained on a general/student cohort. NHSL psychiatric patients have a "
                          "different base rate and case mix; recalibrate on site data before "
                          "interpreting absolute probabilities."),
}
(ARTEFACTS / "dcar_metadata.json").write_text(json.dumps(metadata, indent=2))
np.save(ARTEFACTS / "dcar_reference_scores.npy", ref_scores)

print(json.dumps(metadata, indent=2))
print("\nartefacts:", [p.name for p in sorted(ARTEFACTS.iterdir())])

## 15 · Reporting checklist for the paper

Copy these into §III-E (methods) and §IV (results):

- [ ] Cohort: source, n after each exclusion, GAD-7 distribution, Cronbach's alpha
- [ ] Instrument integrity: item-range violations, `score` vs recomputed-sum mismatch rate
- [ ] Splits: 60/20/20, stratified, calibration split used only for calibration and threshold selection
- [ ] Model comparison table (dummy → GLM → HGB), with the CV protocol stated
- [ ] **Permutation null** with p-value — for every headline metric
- [ ] Test metrics with bootstrap 95% CIs, and prevalence alongside AUPRC
- [ ] Calibration: reliability diagram, Brier, ECE, before and after isotonic
- [ ] Ordinal quality: quadratic-weighted kappa, expected-GAD7 MAE in score points
- [ ] Feature ablation table — this is the justification for the final variable set
- [ ] Subgroup table by gender and age band, **including the slices where it performs worst**
- [ ] Explicit statement that the model is a **population prior**, not a diagnostic instrument
- [ ] Explicit population-transfer caveat (student cohort → psychiatric cohort)
- [ ] Note that `time1..time7` were excluded, and why

**One thing to be honest about in the discussion.** If your AUROC lands around 0.63 with a permutation null at 0.50, the correct sentence is: *"Demographic and lifestyle characteristics carried a small but statistically reliable association with GAD-7 severity (AUROC 0.63, 95% CI …, permutation null 0.50, p < .001), consistent with the modest effect sizes reported for sociodemographic predictors of anxiety. The component is therefore deployed as a population prior within the fusion layer rather than as a standalone screening instrument."* That sentence survives a viva. "Our model achieves 0.63 AUROC for anxiety detection" does not.